# Football Match Predictions — Retrain Model

This notebook retrains the neural network used by the Streamlit app with **corrected features** (no data leakage).

After training, it exports `model_weights.npz` which you download and push to the GitHub repo.

**Run all cells in order.** Training takes about 1-2 minutes.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q tensorflow pandas scikit-learn numpy

In [ ]:
# Cell 2 — Load historical + current season data

import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import datetime
import math
import warnings
warnings.filterwarnings('ignore')

HISTORICAL_CSV_URL = 'https://raw.githubusercontent.com/mundoc/Football-Match-Predict/main/light_leagues_data23_24.csv'

matches_old = pd.read_csv(HISTORICAL_CSV_URL)
season_dfs = [matches_old]

for scode in ['2425', '2526']:
    for div in ['SP1', 'E0']:
        try:
            url = f'https://www.football-data.co.uk/mmz4281/{scode}/{div}.csv'
            df = pd.read_csv(url)
            if not df.empty:
                season_dfs.append(df)
                print(f'  Loaded {scode}/{div}: {len(df)} matches')
        except Exception as e:
            print(f'  Skipped {scode}/{div}: {e}')

matches = pd.concat(season_dfs, axis=0, ignore_index=True)
matches = matches.dropna(subset=['Date'])
matches['Date'] = pd.to_datetime(matches['Date'], format='mixed', dayfirst=True)
matches['Season'] = np.where(
    matches['Date'].dt.month >= 8,
    matches['Date'].dt.year,
    matches['Date'].dt.year - 1,
)
matches['Season'] = (
    matches['Season'].astype(str) + '-'
    + (matches['Season'] + 1).astype(str).str[-2:]
)

FD_NAME_MAP = {
    'Villareal': 'Villarreal', 'Celta Vigo': 'Celta',
    'CA Osasuna': 'Osasuna', 'Atlético': 'Ath Madrid',
    'Athletic Club': 'Ath Bilbao', 'Almería': 'Almeria',
    'Rayo Vallecano': 'Vallecano', 'Granada CF': 'Granada',
    'Cádiz CF': 'Cadiz', 'Alavés': 'Alaves',
    'Leganés': 'Leganes', 'Real Sociedad': 'Sociedad',
    'Real Betis': 'Betis', 'Real Mallorca': 'Mallorca',
    'Real Valladolid': 'Valladolid', 'Espanyol': 'Espanol',
    'Wolverhampton Wanderers': 'Wolves',
    'Tottenham Hotspur': 'Tottenham',
    'Brighton and Hove Albion': 'Brighton',
    'Nottm Forest': \"Nott'm Forest\",
    'Newcastle United': 'Newcastle',
    'West Ham United': 'West Ham',
    'Man Utd': 'Man United', 'Luton Town': 'Luton',
    'Ipswich': 'Ipswich Town',
}
matches = matches.replace(FD_NAME_MAP)
matches['Date'] = pd.to_datetime(matches['Date'], format='mixed', dayfirst=True)
matches.sort_values(by='Date', inplace=True)

print(f'\nTotal matches loaded: {len(matches)}')
print(f'Date range: {matches["Date"].min().date()} to {matches["Date"].max().date()}')
print(f'Seasons: {sorted(matches["Season"].unique())}')

In [ ]:
# Cell 3 — Feature engineering (with leakage fixes)

FEATURE_COLS = [
    'LastEncounterLost', 'LastEncounterWon',
    'HomeChampionsLeague', 'AwayChampionsLeague',
    'HomeEuropaLeague', 'AwayEuropaLeague',
    'reds_last_game_home', 'reds_last_game_away',
    'Rolling_HY', 'Rolling_HS', 'Rolling_HST', 'Rolling_HC',
    'Rolling_AY', 'Rolling_AS', 'Rolling_AST', 'Rolling_AC',
    'HomeTeam_RecentGoalDiff', 'AwayTeam_RecentGoalDiff',
    'HomeTeam_RecentPoints', 'AwayTeam_RecentPoints',
    'HomeTeam_PointsThisSeason', 'AwayTeam_PointsThisSeason',
    'HomeRanking', 'AwayRanking', 'SP1', 'E0',
]

HISTORICAL_RANKINGS = {
    'Barcelona': 1, 'Real Madrid': 2, 'Ath Madrid': 3, 'Sevilla': 4,
    'Valencia': 5, 'Villarreal': 6, 'Ath Bilbao': 7, 'Sociedad': 8,
    'Getafe': 9, 'Espanol': 10, 'Betis': 11, 'Osasuna': 12,
    'Celta': 13, 'Levante': 14, 'Mallorca': 15, 'Malaga': 16,
    'La Coruna': 17, 'Granada': 18, 'Valladolid': 19, 'Vallecano': 20,
    'Zaragoza': 21, 'Santander': 22, 'Eibar': 23, 'Alaves': 24,
    'Almeria': 25, 'Sp Gijon': 26, 'Elche': 27, 'Cadiz': 28,
    'Leganes': 29, 'Girona': 30, 'Recreativo': 31, 'Las Palmas': 32,
    'Huesca': 33, 'Tenerife': 34, 'Hercules': 35, 'Numancia': 36,
    'Murcia': 37, 'Xerez': 38, 'Gimnastic': 39, 'Cordoba': 40,
    'Oviedo': 41,
    'Man United': 1, 'Liverpool': 2, 'Man City': 3, 'Chelsea': 4,
    'Arsenal': 5, 'Tottenham': 6, 'Everton': 7, 'West Ham': 8,
    'Newcastle': 9, 'Aston Villa': 10, 'Fulham': 11, 'Southampton': 12,
    'Leicester': 13, 'Crystal Palace': 14, 'Wolves': 15, 'Stoke': 16,
    'Sunderland': 17, 'West Brom': 18, 'Blackburn': 19, 'Wigan': 20,
    'Burnley': 21, 'Bolton': 22, 'Brighton': 23, 'Swansea': 24,
    'Bournemouth': 25, 'Watford': 26, 'Portsmouth': 27, 'Norwich': 28,
    'Middlesbrough': 29, 'Hull': 30, 'Birmingham': 31, 'Leeds': 32,
    'Brentford': 33, 'Sheffield United': 34, 'Reading': 35,
    'Cardiff': 36, 'QPR': 37, 'Charlton': 38, \"Nott'm Forest\": 39,
    'Blackpool': 40, 'Huddersfield': 41, 'Luton': 42, 'Derby': 43,
    'Ipswich Town': 44,
}


def assign_points(row):
    if row['FTR'] == 'H':
        return (row['HomeTeam'], 3), (row['AwayTeam'], 0)
    elif row['FTR'] == 'A':
        return (row['HomeTeam'], 0), (row['AwayTeam'], 3)
    return (row['HomeTeam'], 1), (row['AwayTeam'], 1)


def create_rolling_sum(df, team_col, feature_col):
    temp = df.copy().sort_values(by=[team_col, 'Date'])
    col_name = f'Rolling_{feature_col}_{team_col}'
    temp[col_name] = (
        temp.groupby(team_col)[feature_col]
        .rolling(window=5, min_periods=1).sum()
        .shift().reset_index(level=0, drop=True)
    )
    return temp[col_name]


# --- 1. Recent form (5 & 20 games) ---
tp5 = defaultdict(lambda: np.zeros(5, dtype=int))
tp20 = defaultdict(lambda: np.zeros(20, dtype=int))
tgd = defaultdict(lambda: np.zeros(5, dtype=int))

for idx, row in matches.iterrows():
    ht, at = row['HomeTeam'], row['AwayTeam']
    hp, ap = assign_points(row)
    hgd = row['FTHG'] - row['FTAG']
    agd = row['FTAG'] - row['FTHG']
    matches.at[idx, 'HomeTeam_RecentPoints'] = tp5[ht].sum()
    matches.at[idx, 'AwayTeam_RecentPoints'] = tp5[at].sum()
    matches.at[idx, 'HomeTeam_RecentGoalDiff'] = tgd[ht].sum()
    matches.at[idx, 'AwayTeam_RecentGoalDiff'] = tgd[at].sum()
    matches.at[idx, 'HomeTeam_RecentPoints20'] = tp20[ht].sum()
    matches.at[idx, 'AwayTeam_RecentPoints20'] = tp20[at].sum()
    for arr, team, val in [
        (tp5, ht, hp[1]), (tp5, at, ap[1]),
        (tp20, ht, hp[1]), (tp20, at, ap[1]),
    ]:
        arr[team] = np.roll(arr[team], -1); arr[team][-1] = val
    for team, val in [(ht, hgd), (at, agd)]:
        tgd[team] = np.roll(tgd[team], -1)
        tgd[team][-1] = val if not math.isnan(val) else 0

print('Recent form features computed.')

# --- 2. Last encounter ---
ler = defaultdict(lambda: 'NA')
for h in matches['HomeTeam'].unique():
    for a in matches['AwayTeam'].unique():
        ler[(h, a)] = 'D'
for idx, row in matches.iterrows():
    t = (row['HomeTeam'], row['AwayTeam'])
    r = (row['AwayTeam'], row['HomeTeam'])
    matches.at[idx, 'LastEncounterWon'] = 0
    matches.at[idx, 'LastEncounterLost'] = 0
    if ler[t] == 'H':
        matches.at[idx, 'LastEncounterWon'] = 1
    elif ler[t] == 'A':
        matches.at[idx, 'LastEncounterLost'] = 1
    elif ler[r] == 'A':
        matches.at[idx, 'LastEncounterWon'] = 1
    elif ler[r] == 'H':
        matches.at[idx, 'LastEncounterLost'] = 1
    ler[t] = row['FTR']

print('Last encounter features computed.')

# --- 3. European competition flags (FIX: use PREVIOUS season) ---
positions_per_season = pd.DataFrame(
    columns=['Season', 'Team', 'Wins', 'Draws', 'Points', 'Position'])
tps = defaultdict(lambda: defaultdict(int))
tws = defaultdict(lambda: defaultdict(int))
tds_dict = defaultdict(lambda: defaultdict(int))
for _, row in matches.iterrows():
    ht, at = row['HomeTeam'], row['AwayTeam']
    hp, ap = assign_points(row)
    tps[row['Season']][ht] += hp[1]
    tps[row['Season']][at] += ap[1]
    if hp[1] == 3:
        tws[row['Season']][ht] += 1
    elif ap[1] == 3:
        tws[row['Season']][at] += 1
    else:
        tds_dict[row['Season']][ht] += 1
        tds_dict[row['Season']][at] += 1
for season, team_pts in tps.items():
    sd = {'Season': season, 'Team': [], 'Wins': [], 'Draws': [],
          'Points': [], 'Position': [], 'Div': []}
    sorted_teams = sorted(team_pts.keys(),
                          key=lambda x: (team_pts[x], x), reverse=True)
    for pos, team in enumerate(sorted_teams, 1):
        team_div = matches[
            (matches['Season'] == season)
            & ((matches['HomeTeam'] == team) | (matches['AwayTeam'] == team))
        ]['Div'].values[0]
        sd['Team'].append(team)
        sd['Wins'].append(tws[season][team])
        sd['Draws'].append(tds_dict[season][team])
        sd['Points'].append(team_pts[team])
        sd['Position'].append(pos)
        sd['Div'].append(team_div)
    positions_per_season = pd.concat(
        [positions_per_season, pd.DataFrame(sd)], ignore_index=True)

for col in ['HomeChampionsLeague', 'AwayChampionsLeague',
            'HomeEuropaLeague', 'AwayEuropaLeague']:
    matches[col] = 0

first_season = '2005-2006'
cl = {'SP1': ['Barcelona', 'Real Madrid', 'Betis', 'Villarreal'],
      'E0': ['Chelsea', 'Arsenal', 'Man United', 'Everton']}
el = {'SP1': ['Espanol', 'Sevilla'], 'E0': ['Liverpool', 'Bolton']}
for div, teams in cl.items():
    mask = (matches['Season'] == first_season) & (matches['Div'] == div)
    matches.loc[mask, 'HomeChampionsLeague'] = matches['HomeTeam'].isin(teams).astype(int)
    matches.loc[mask, 'AwayChampionsLeague'] = matches['AwayTeam'].isin(teams).astype(int)
for div, teams in el.items():
    mask = (matches['Season'] == first_season) & (matches['Div'] == div)
    matches.loc[mask, 'HomeEuropaLeague'] = matches['HomeTeam'].isin(teams).astype(int)
    matches.loc[mask, 'AwayEuropaLeague'] = matches['AwayTeam'].isin(teams).astype(int)

def prev_season_str(s):
    year = int(s[:4])
    return f'{year - 1}-{str(year)[-2:]}'

for idx, row in matches.iterrows():
    if row['Season'] == first_season:
        continue
    prev_s = prev_season_str(row['Season'])
    div = row['Div']
    for prefix, team_col in [('Home', 'HomeTeam'), ('Away', 'AwayTeam')]:
        pos = positions_per_season[
            (positions_per_season['Season'] == prev_s)
            & (positions_per_season['Team'] == row[team_col])
            & (positions_per_season['Div'] == div)
        ]['Position'].values
        if pos.size > 0:
            matches.at[idx, f'{prefix}ChampionsLeague'] = int(1 <= pos[0] <= 4)
            matches.at[idx, f'{prefix}EuropaLeague'] = int(5 <= pos[0] <= 6)

print('European competition flags computed (using previous season).')

# --- 4. Historical rankings ---
matches['HomeRanking'] = matches['HomeTeam'].map(HISTORICAL_RANKINGS)
matches['AwayRanking'] = matches['AwayTeam'].map(HISTORICAL_RANKINGS)

# --- 5. Red cards last game ---
matches = matches.sort_values(by='Date')
lgr = {}
matches['reds_last_game_home'] = 0
matches['reds_last_game_away'] = 0
for idx, row in matches.iterrows():
    matches.at[idx, 'reds_last_game_home'] = lgr.get(row['HomeTeam'], 0)
    matches.at[idx, 'reds_last_game_away'] = lgr.get(row['AwayTeam'], 0)
    lgr[row['HomeTeam']] = row.get('HR', 0)
    lgr[row['AwayTeam']] = row.get('AR', 0)

# --- 6. Rolling match stats ---
for feat in ['HY', 'HS', 'HST', 'HC', 'AY', 'AS', 'AST', 'AC']:
    tt = 'HomeTeam' if feat.startswith('H') else 'AwayTeam'
    matches[f'Rolling_{feat}'] = create_rolling_sum(matches, tt, feat)
matches.fillna(0, inplace=True)

print('Rolling stats computed.')

# --- 7. Points this season (FIX: store BEFORE updating) ---
matches['MatchOutcome'] = matches['FTR'].map({'H': 2, 'D': 1, 'A': 0})
matches = matches[matches['Date'].dt.year != 2005]
matches = matches.sort_values(by='Date')
tpcs = defaultdict(int)
cur_s = None
for idx, row in matches.iterrows():
    if cur_s != row['Season']:
        cur_s = row['Season']
        tpcs = defaultdict(int)
    ht, at = row['HomeTeam'], row['AwayTeam']
    matches.at[idx, 'HomeTeam_PointsThisSeason'] = tpcs[ht]
    matches.at[idx, 'AwayTeam_PointsThisSeason'] = tpcs[at]
    hp, ap = assign_points(row)
    tpcs[ht] += hp[1]
    tpcs[at] += ap[1]

# --- 8. League dummies ---
matches['SP1'] = (matches['Div'] == 'SP1').astype(int)
matches['E0'] = (matches['Div'] == 'E0').astype(int)

# --- 9. Drop rows with missing outcome ---
matches = matches[matches['MatchOutcome'].notna()].copy()
matches = matches.sort_values('Date')

print(f'\nFeature engineering complete. {len(matches)} matches with outcomes.')
print(f'Features: {len(FEATURE_COLS)}')

In [ ]:
# Cell 4 — Train the model (temporal split, same architecture)

import tensorflow as tf
from sklearn.preprocessing import StandardScaler

X = matches[FEATURE_COLS].values
y = matches['MatchOutcome'].values.astype(int)

# Temporal split: oldest 60% train, next 20% val, last 20% test
n = len(X)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Train period: up to {matches.iloc[train_end-1]["Date"].date()}')
print(f'Test period: from {matches.iloc[val_end]["Date"].date()}')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Convert labels to one-hot for categorical crossentropy
y_train_oh = tf.keras.utils.to_categorical(y_train, 3)
y_val_oh = tf.keras.utils.to_categorical(y_val, 3)

# Build the same architecture the app's nn_predict() expects:
# Dense(128,relu) -> BN -> Dropout(0.3)
# Dense(64,relu)  -> BN -> Dropout(0.3)
# Dense(32,relu)  -> BN -> Dropout(0.3)
# Dense(3,softmax)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(26,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

history = model.fit(
    X_train_s, y_train_oh,
    validation_data=(X_val_s, y_val_oh),
    epochs=50,
    batch_size=64,
    verbose=1,
)

print(f'\nFinal val accuracy: {history.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Cell 5 — Evaluate on the held-out test set

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_probs = model.predict(X_test_s)
y_pred = y_pred_probs.argmax(axis=1)

labels = ['Away Win (0)', 'Draw (1)', 'Home Win (2)']
print('Classification Report (temporal test set):\n')
print(classification_report(y_test, y_pred, target_names=labels))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Temporal Test Set')
plt.tight_layout()
plt.show()

accuracy = (y_pred == y_test).mean()
print(f'\nTest accuracy: {accuracy:.4f} ({(y_pred == y_test).sum()}/{len(y_test)})')

In [ ]:
# Cell 6 — Export weights and download
#
# The keys must match what nn_predict() in the app expects:
#   dense_kernel, dense_bias,
#   batch_normalization_gamma, batch_normalization_beta,
#   batch_normalization_moving_mean, batch_normalization_moving_variance,
#   dense_1_kernel, ..., dense_3_kernel, dense_3_bias

weights = {}

dense_idx = 0
bn_idx = 0

for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        name = 'dense' if dense_idx == 0 else f'dense_{dense_idx}'
        kernel, bias = layer.get_weights()
        weights[f'{name}_kernel'] = kernel
        weights[f'{name}_bias'] = bias
        dense_idx += 1
    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        name = 'batch_normalization' if bn_idx == 0 else f'batch_normalization_{bn_idx}'
        gamma, beta, moving_mean, moving_variance = layer.get_weights()
        weights[f'{name}_gamma'] = gamma
        weights[f'{name}_beta'] = beta
        weights[f'{name}_moving_mean'] = moving_mean
        weights[f'{name}_moving_variance'] = moving_variance
        bn_idx += 1

np.savez_compressed('model_weights.npz', **weights)
print(f'Saved model_weights.npz ({len(weights)} arrays)')
print('Keys:', sorted(weights.keys()))

# --- Verify the exported weights work with the app's forward pass ---
def nn_predict_test(X_input):
    w = np.load('model_weights.npz')
    eps = 1e-3
    def dense(x, layer): return x @ w[f'{layer}_kernel'] + w[f'{layer}_bias']
    def batch_norm(x, layer):
        return w[f'{layer}_gamma'] * (x - w[f'{layer}_moving_mean']) / np.sqrt(w[f'{layer}_moving_variance'] + eps) + w[f'{layer}_beta']
    def relu(x): return np.maximum(0, x)
    def softmax(x):
        e = np.exp(x - x.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)
    x = relu(dense(X_input, 'dense'))
    x = batch_norm(x, 'batch_normalization')
    x = relu(dense(x, 'dense_1'))
    x = batch_norm(x, 'batch_normalization_1')
    x = relu(dense(x, 'dense_2'))
    x = batch_norm(x, 'batch_normalization_2')
    x = softmax(dense(x, 'dense_3'))
    return x

np_preds = nn_predict_test(X_test_s[:5])
tf_preds = model.predict(X_test_s[:5], verbose=0)
print(f'\nNumpy vs TF match: {np.allclose(np_preds, tf_preds, atol=1e-5)}')

# --- Download the file ---
try:
    from google.colab import files
    files.download('model_weights.npz')
    print('\nDownload started! Replace the file in your repo and push to GitHub.')
except ImportError:
    print('\nNot running in Colab. File saved to current directory.')